# Sumativa 3 — Parte 4: impacto de la estrategia de imputación

Esta sección ajusta **la misma especificación logística** bajo eliminación, imputación simple e imputación por regresión. Se comparan tamaño muestral, coeficientes, intervalos de confianza, AUC y accuracy en prueba. No selecciona variables ni reemplaza el trabajo de modelamiento del grupo; solamente mide si el tratamiento de faltantes modifica sus conclusiones.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.metrics import roc_curve

from utils_s3 import (
    cargar_datos, preparar_particiones_por_estrategia, evaluar_modelo_comun
)

BASE = Path('.')
TABLAS = BASE / 'tablas'
FIGURAS = BASE / 'figuras'
TABLAS.mkdir(exist_ok=True)
FIGURAS.mkdir(exist_ok=True)
sns.set_theme(style='whitegrid', context='notebook')

## 4.1 Especificación común y prevención de fuga

La lista siguiente es el único punto de integración con la Parte 2 del grupo. Si existe `config_modelo_final.json`, se usan sus variables; en caso contrario se utiliza una especificación operativa basada en los insumos robustos S1/S2. Cuando el grupo seleccione definitivamente su modelo, basta reemplazar esa lista o agregar el archivo de configuración, sin modificar el resto del notebook.

La división 70/30 se realiza **antes** de imputar. Medianas, modas y regresiones se aprenden exclusivamente en entrenamiento y se aplican después a prueba. Esto cumple la recomendación metodológica de S2 y evita fuga de información.

In [ ]:
config = BASE / 'config_modelo_final.json'
if config.exists():
    VARIABLES_MODELO_FINAL = json.loads(config.read_text(encoding='utf-8'))['variables']
    fuente_modelo = 'config_modelo_final.json aportado por la Parte 2'
else:
    VARIABLES_MODELO_FINAL = ['age', 'male', 'cigsPerDay', 'sysBP', 'glucose']
    fuente_modelo = 'configuración operativa S1/S2; actualizar al integrar la Parte 2'
print('Variables:', VARIABLES_MODELO_FINAL)
print('Fuente:', fuente_modelo)

In [ ]:
original = cargar_datos()
particiones, diagnosticos, conteos_train, conteos_test = preparar_particiones_por_estrategia(original)

metricas = []
coeficientes = []
curvas = []
for estrategia, (train, test) in particiones.items():
    modelo, fila, tabla_coef, probabilidades = evaluar_modelo_comun(
        train, test, VARIABLES_MODELO_FINAL, estrategia
    )
    metricas.append(fila)
    coeficientes.append(tabla_coef)
    curvas.append((estrategia, test['TenYearCHD'].to_numpy(), probabilidades))

metricas = pd.DataFrame(metricas)
coeficientes = pd.concat(coeficientes, ignore_index=True)
metricas.to_csv(TABLAS / 'p4_impacto_imputacion_metricas.csv', index=False)
coeficientes.to_csv(TABLAS / 'p4_impacto_imputacion_coeficientes.csv', index=False)
display(metricas.style.format({
    'accuracy': '{:.3f}', 'precision': '{:.3f}', 'recall': '{:.3f}',
    'F1': '{:.3f}', 'AUC': '{:.3f}'
}))
display(coeficientes.style.format({
    'coeficiente': '{:.3f}', 'IC95_inf': '{:.3f}', 'IC95_sup': '{:.3f}',
    'odds_ratio': '{:.3f}', 'IC95_OR_inf': '{:.3f}',
    'IC95_OR_sup': '{:.3f}', 'p_valor': '{:.3g}'
}))

Los predictores continuos se estandarizan con media y desviación estándar de cada entrenamiento. Por tanto, sus odds ratios representan un incremento de una desviación estándar y son comparables entre estrategias. Las variables binarias conservan su escala 0/1. El umbral 0,5 se mantiene fijo para que accuracy y matriz de confusión respondan solo al cambio de datos y no a diferentes reglas de clasificación.

In [ ]:
coef_sin_const = coeficientes.query("variable != 'const'").copy()
sensibilidad = []
for variable, grupo in coef_sin_const.groupby('variable'):
    sensibilidad.append({
        'variable': variable,
        'signo_consistente': grupo['coeficiente'].gt(0).nunique() == 1,
        'significativo_en_las_tres': grupo['p_valor'].lt(0.05).all(),
        'OR_min': grupo['odds_ratio'].min(),
        'OR_max': grupo['odds_ratio'].max(),
        'amplitud_OR': grupo['odds_ratio'].max() - grupo['odds_ratio'].min(),
    })
sensibilidad = pd.DataFrame(sensibilidad)
sensibilidad.to_csv(TABLAS / 'p4_estabilidad_conclusiones.csv', index=False)
display(sensibilidad.style.format({'OR_min': '{:.3f}', 'OR_max': '{:.3f}', 'amplitud_OR': '{:.3f}'}))

rango_auc = metricas['AUC'].max() - metricas['AUC'].min()
rango_accuracy = metricas['accuracy'].max() - metricas['accuracy'].min()
print(f'Rango de AUC entre estrategias: {rango_auc:.4f}')
print(f'Rango de accuracy entre estrategias: {rango_accuracy:.4f}')
print('¿Todos los efectos conservan signo?', sensibilidad['signo_consistente'].all())
print('¿Todos conservan significación al 5 %?', sensibilidad['significativo_en_las_tres'].all())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
metricas.plot(x='estrategia', y=['AUC', 'accuracy'], kind='bar', ylim=(0, 1), ax=axes[0],
              color=['#2A6F97', '#E09F3E'])
axes[0].set(title='Desempeño en prueba', xlabel='', ylabel='Valor')
axes[0].tick_params(axis='x', rotation=0)
for estrategia, y, p in curvas:
    fpr, tpr, _ = roc_curve(y, p)
    auc = metricas.loc[metricas['estrategia'] == estrategia, 'AUC'].iloc[0]
    axes[1].plot(fpr, tpr, lw=2, label=f'{estrategia} (AUC={auc:.3f})')
axes[1].plot([0, 1], [0, 1], '--', color='0.5')
axes[1].set(title='Curvas ROC', xlabel='1 - especificidad', ylabel='Sensibilidad')
axes[1].legend()
fig.tight_layout()
plt.savefig(FIGURAS / 'p4_fig1_metricas_roc_imputacion.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
sin_constante = coeficientes.query("variable != 'const'").copy()
variables = sin_constante['variable'].unique().tolist()
estrategias = ['eliminacion', 'simple', 'regresion']
desplazamientos = {'eliminacion': -0.18, 'simple': 0.0, 'regresion': 0.18}
colores = {'eliminacion': '#6C757D', 'simple': '#E09F3E', 'regresion': '#2A6F97'}
fig, ax = plt.subplots(figsize=(8, max(4, 0.7 * len(variables))))
for i, variable in enumerate(variables):
    for estrategia in estrategias:
        fila = sin_constante.query('variable == @variable and estrategia == @estrategia').iloc[0]
        y = i + desplazamientos[estrategia]
        ax.errorbar(fila['odds_ratio'], y,
                    xerr=[[fila['odds_ratio']-fila['IC95_OR_inf']],
                          [fila['IC95_OR_sup']-fila['odds_ratio']]],
                    fmt='o', color=colores[estrategia], capsize=3, label=estrategia if i == 0 else None)
ax.axvline(1, color='0.4', linestyle='--')
ax.set_yticks(range(len(variables)), variables)
ax.set(xlabel='Odds ratio e IC 95 %', title='Estabilidad de efectos según estrategia de faltantes')
ax.legend()
fig.tight_layout()
plt.savefig(FIGURAS / 'p4_fig2_or_imputacion.png', dpi=180, bbox_inches='tight')
plt.show()

## 4.2 Interpretación y recomendación

La comparación debe separar dos aspectos. Primero, eliminación utiliza menos personas y menos eventos; por ello su AUC puede fluctuar por composición muestral y no representa automáticamente un método superior. Segundo, imputación simple y por regresión se evalúan sobre exactamente las mismas filas de prueba, de modo que sus diferencias reflejan directamente el tratamiento de faltantes.

La conclusión se considera robusta cuando los coeficientes mantienen signo, los intervalos presentan amplio solapamiento y AUC/accuracy cambian poco. Si una variable cambia de signo, cruza el valor nulo solo bajo una estrategia o altera materialmente el AUC, deberá informarse como sensible a imputación. La recomendación principal es conservar la imputación por regresión por trazabilidad con S1/S2 y tamaño muestral, acompañándola siempre con esta tabla de sensibilidad.

**Punto de integración grupal:** antes de entregar, verificar que `VARIABLES_MODELO_FINAL` coincida exactamente con el modelo elegido en la Parte 2. Ningún otro bloque necesita cambios.